In [1]:
from tinygrad.nn.datasets import mnist

X_train, Y_train, X_test, Y_test = mnist()
print(X_train.shape, X_train.dtype, Y_train.shape, Y_train.dtype)

(60000, 1, 28, 28) dtypes.uchar (60000,) dtypes.uchar


In [2]:
from tinygrad import Tensor, dtypes, nn

class Model:
    def __init__(self):
        self.l1 = nn.Conv2d(1, 64, (3, 3))
        self.l2 = nn.Conv2d(64, 128, (3, 3))
        self.l3 = nn.Linear(3200, 10)

    def __call__(self, x: Tensor)-> Tensor:
        x = self.l1(x).relu()
        x = x.max_pool2d((2, 2))
        x = self.l2(x).relu()
        x = x.max_pool2d((2, 2))
        return self.l3(x.flatten(1).dropout(0.5))

model = Model()

acc = (model(X_test).argmax(1) == Y_test).mean()
print(f"Accuracy: {acc.item():.4f}")


Accuracy: 0.0656


In [21]:
optim = nn.optim.Adam(nn.state.get_parameters(model))

batch_size = 128

def step():
    Tensor.training = True
    samples = Tensor.randint(batch_size, high=X_train.shape[0])
    X, Y = X_train[samples], Y_train[samples]
    optim.zero_grad()
    loss = model(X).sparse_categorical_crossentropy(Y)
    loss.backward()
    optim.step()
    return loss



In [22]:
import timeit

timeit.repeat(step, repeat=5, number=1)

[0.05828562402166426,
 0.05034835392143577,
 0.0503022939665243,
 0.04984218394383788,
 0.04964962403755635]

In [24]:
from tinygrad import GlobalCounters, Context
GlobalCounters.reset()
with Context(DEBUG=4):
    step()

split 40: (128, 28, 28, 60000, 1) -> (128, 28, 28, 1500, 1, 40) -> (128, 28, 28, 1, 1)
split 250: (128, 60000) -> (128, 240, 250) -> (128, 1)
split 128: (64, 3, 3, 128, 1, 26, 26, 1) -> (64, 3, 3, 1, 1, 26, 26, 1, 128) -> (64, 3, 3, 1, 1, 1, 1, 1)
split 128: (64, 128, 26, 26) -> (64, 1, 26, 26, 128) -> (64, 1, 1, 1)
scheduled 63 kernels in 41.39 ms
*** NV         1 E_n18                                        arg  1 mem  0.16 GB tm      1.86us/     0.00ms (     0.00 GFLOPS    0.0|0.0     GB/s) ['__imul__']
*** NV         2 E_n19                                        arg  1 mem  0.16 GB tm      1.82us/     0.00ms (     0.00 GFLOPS    0.0|0.0     GB/s) ['__imul__']
*** NV         3 r_64_16_4                                    arg  1 mem  0.16 GB tm      1.57us/     0.01ms (    16.33 GFLOPS    0.2|47.0    GB/s) ['randint']
*** NV         4 E_n12                                        arg  1 mem  0.16 GB tm      1.82us/     0.01ms (     0.00 GFLOPS    0.0|0.0     GB/s) ['randint']
*** NV 

In [8]:
from tinygrad import TinyJit
jit_step = TinyJit(step)

In [9]:
import timeit

timeit.repeat(jit_step, repeat=5, number=1)

[0.13410261506214738,
 0.05642486293800175,
 0.0045639240415766835,
 0.0025121619692072272,
 0.0026831229915842414]

In [31]:
with Context(DEBUG=4):
    for step in range(7000):
        loss = jit_step()

        if step % 100 == 0:
            Tensor.training = False
            acc = (model(X_test).argmax(axis=-1) == Y_test).mean().item()
            print(f"step={step} loss={loss.item():.2f} acc: {acc * 100.:.2f}%")
            break

*** NV       124 <batched 32>                                 arg  0 mem  0.16 GB tm    919.58us/   350.02ms (  2762.57 GFLOPS  201.4|2757.1  GB/s) 
*** NV       125 <batched 34>                                 arg  0 mem  0.16 GB tm   1770.96us/   351.79ms (  2680.41 GFLOPS  147.2|2845.9  GB/s) 
scheduled 8 kernels in 7.22 ms
*** NV       126 r_2500_13_13_16_2_2_4_4_3_3                  arg  4 mem  1.89 GB tm   4442.94us/   356.24ms (  1752.78 GFLOPS  391.3|1582.4  GB/s) ['conv2d']
*** NV       127 r_65000_13_32_4_2_2                          arg  2 mem  2.32 GB tm   3002.98us/   359.24ms (   504.25 GFLOPS  720.4|720.4   GB/s) ['max_pool2d', 'relu']
*** NV       128 r_625_2_11_11_4_16_4_4_64_3_3                arg  4 mem  1.21 GB tm     46.37ms/   405.60ms (  3851.51 GFLOPS   22.7|3864.9  GB/s) ['conv2d']
*** NV       129 r_10000_5_5_32_4_2_2                         arg  2 mem  0.91 GB tm   1078.62us/   406.68ms (   415.34 GFLOPS  593.3|593.3   GB/s) ['max_pool2d', 'relu']
*** NV     